Historical study notebook. Its original input is unavailable in this public edition, so this is not a runnable public demo. It may require an input with the original structure and dependencies. This work used supplied learning resources and starter material; retained attribution appears with the notebook content.



In [ ]:
!pip install -q llama-index llama-index-embeddings-huggingface langchain langchain-text-splitters llama-cpp-python llama-index-llms-llama-cpp PyPDF2

In [ ]:
from PyPDF2 import PdfReader
reader = PdfReader("your-document.pdf")
pages = [{"page": i, "text": p.extract_text() or "",
          "doc_type": "Packaging Specification",   # you can reuse your Step-1 classifier, or label simply
          "doc_id": "pharma_blob_01",
          "source_file": "your-document.pdf",
          "page_in_doc": i}
         for i, p in enumerate(reader.pages)]

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
full_text = "\n\n".join(p["text"] for p in pages)
splitter = RecursiveCharacterTextSplitter(chunk_size=512, chunk_overlap=100)
chunks = splitter.split_text(full_text)

In [ ]:
from llama_index.core import Document
all_documents = [
    Document(text=chunk, metadata={
        "doc_type": "Packaging Specification",
        "chunk_index": i,
        "doc_id": "pharma_blob_01",
        "source_file": "your-document.pdf"
    })
    for i, chunk in enumerate(chunks)
]

In [ ]:
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.llms.llama_cpp import LlamaCPP
from llama_index.core import VectorStoreIndex, Settings

Settings.embed_model = HuggingFaceEmbedding(model_name="BAAI/bge-small-en-v1.5")
Settings.llm = LlamaCPP(
    model_url="https://huggingface.co/TheBloke/Mistral-7B-Instruct-v0.2-GGUF/resolve/main/mistral-7b-instruct-v0.2.Q4_K_M.gguf",
    temperature=0.1, max_new_tokens=256, context_window=3900,
    model_kwargs={"n_gpu_layers": -1}, verbose=False,
)
index = VectorStoreIndex.from_documents(all_documents)

In [ ]:
from llama_index.core.vector_stores import MetadataFilters, MetadataFilter, FilterOperator
query = "Were there any packaging configuration changes?"
retriever = index.as_retriever(
    similarity_top_k=4,
    filters=MetadataFilters(filters=[
        MetadataFilter(key="doc_type", value="Packaging Specification", operator=FilterOperator.EQ)
    ])
)
retrieved_nodes = retriever.retrieve(query)


In [ ]:
from llama_index.core.response_synthesizers import get_response_synthesizer, ResponseMode
synthesizer = get_response_synthesizer(response_mode=ResponseMode.COMPACT)
response = synthesizer.synthesize(query, nodes=retrieved_nodes)
print("Answer:", response.response)


In [ ]:
import json
result = {
    "query": query,
    "predicted_doc_type": "Packaging Specification",
    "matched_chunks": [{"text": n.text, "metadata": n.metadata} for n in retrieved_nodes],
    "answer": response.response
}
print(json.dumps(result, indent=2))
